# Guided Practical: How Much Electricity Can a City's Sunshine Produce?

In a dense urban neighborhood, is the largest roof always the best place for PV? Alongside roof area, we need to understand the weather, the Sun's position, and nearby buildings.

This practical follows **city and buildings → weather → Sun and shadows → light reaching the modules → electricity generation**. You will choose a city, explore its solar resource, and propose an initial PV design for one building.

The example uses Bangkok. Choose Bangkok, Shenzhen, Seoul, or Tokyo from the supplied data.

## Prepare the Course

**Colab:** choose **File > Save a copy in Drive**, then run the cell below. It downloads the course code and all four cities, installs the course libraries and checks the data. Use a CPU runtime. After setup, run the remaining cells in order.

**Local Jupyter:** extract the complete student package and use the supplied launcher. The same cell checks your local course environment.

Save your Notebook regularly. At the end, download the results ZIP; Colab runtime files are temporary.


In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import os
import runpy
import sys
import tempfile
import urllib.request
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    COURSE_URL = "https://raw.githubusercontent.com/Kechuan-ln/roof-pv-course-student/ceeedf9497a6f1a881775f45ee652d2c1d173732/assets/course_runtime.zip"
    COURSE_SHA256 = "5a14b501961ed16434254f53c10bd45d78d446457b985febcc23fd440fc36394"
    course_root = Path('/content') / ('pv_course_' + COURSE_SHA256[:12])
    if not (course_root / '.course_ready').is_file():
        if course_root.exists():
            raise RuntimeError('An incomplete course folder exists. Start a new Colab session and rerun this cell.')
        print('Downloading the course and four-city data...', flush=True)
        with tempfile.TemporaryDirectory(dir='/content') as temporary:
            folder = Path(temporary)
            archive = folder / 'course.zip'
            request = urllib.request.Request(COURSE_URL, headers={'User-Agent': 'PV-Course-Notebook'})
            with urllib.request.urlopen(request, timeout=120) as response, archive.open('wb') as output:
                import shutil
                shutil.copyfileobj(response, output)
            if hashlib.sha256(archive.read_bytes()).hexdigest() != COURSE_SHA256:
                raise RuntimeError('Course download failed its integrity check. Rerun this cell.')
            extracted = folder / 'course'
            extracted.mkdir()
            with zipfile.ZipFile(archive) as package:
                for entry in package.infolist():
                    target = (extracted / entry.filename).resolve()
                    if not target.is_relative_to(extracted.resolve()) or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                        raise RuntimeError('Invalid course archive entry.')
                package.extractall(extracted)
            (extracted / '.course_ready').write_text(COURSE_SHA256)
            extracted.rename(course_root)
    if (course_root / '.course_ready').read_text() != COURSE_SHA256:
        raise RuntimeError('Course version mismatch. Start a new Colab session.')
    ROOT = runpy.run_path(str(course_root / 'scripts/colab_setup.py'))['prepare'](course_root)
else:
    candidates = [candidate for parent in [Path.cwd(), *Path.cwd().parents]
                  for candidate in [parent, parent / 'student']]
    ROOT = next((p for p in candidates if (p / 'pv_course').is_dir()
                 and (p / 'scripts/notebook_setup.py').is_file()), None)
    if ROOT is None:
        raise RuntimeError('Extract the full student package and open PV_Practical.ipynb from that folder.')
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    runpy.run_path(str(ROOT / 'scripts/notebook_setup.py'))['ensure_environment'](ROOT)


## 1. Choose a City and Explore the Data

The weather table records irradiance, air temperature, and wind speed every 10 minutes. The building file contains roof footprints. Keep the default settings first and run the next two cells.

| Setting | Meaning |
|---|---|
| `CITY` | Bangkok / Shenzhen / Seoul / Tokyo |
| `SURFACE_TILT` | Module angle above horizontal: 0° is horizontal; `None` uses the absolute local latitude |
| `SURFACE_AZIMUTH` | Orientation: 90° east, 180° south, 270° west |
| `ROOF_FRACTION` | Assumed fraction of roof area available for modules; default 80% |

For example, a 100 m² roof with an 80% usable fraction provides 80 m² of module area.

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display
from pv_course.pv_workflow import run_assessment
from pv_course.data_quality import interval_energy
import geopandas as gpd
import pvlib
from pv_course.weather_visualization import read_weather
from pv_course.poa_calculation import get_timezone

CITY = "Bangkok"
MODULE_AREA = 1.69  # Area of one module, m²
ROOF_FRACTION = 0.80
SURFACE_TILT = None
SURFACE_AZIMUTH = 180.0
ROOT = Path.cwd()
assert (ROOT / "weather_data").is_dir(), "Open PV_Practical.ipynb with the student folder as the working directory."

# Select an available font for plots.
font_names = {f.name for f in font_manager.fontManager.ttflist}
chinese_fonts = [name for name in ["Microsoft YaHei", "SimHei", "PingFang SC",
    "Heiti TC", "Songti SC", "Noto Sans CJK SC", "WenQuanYi Zen Hei"] if name in font_names]
plt.rcParams.update({"font.family": "sans-serif",
    "font.sans-serif": chinese_fonts + ["DejaVu Sans"], "axes.unicode_minus": False,
    "figure.figsize": (11, 4), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False})
if not chinese_fonts:
    print("Using the default plotting font.")
COLORS = ["#1976A3", "#E69F00", "#009E73"]

In [ ]:
# Load city and weather data first; PV calculations begin in Section 4.
city_buildings = gpd.read_file(ROOT / "buildings_data" / f"{CITY}_building.geojson")
city_local = city_buildings.to_crs(city_buildings.estimate_utm_crs())
site = gpd.GeoSeries([city_local.geometry.unary_union.centroid], crs=city_local.crs).to_crs(4326).iloc[0]
SITE_LAT, SITE_LON = site.y, site.x
weather = read_weather(ROOT / "weather_data" / f"{CITY}_weather.csv")
weather.index = weather.index.tz_localize(get_timezone(SITE_LAT, SITE_LON))
STEP_MINUTES = 10
YEAR = weather.index[0].year
solar_position = pvlib.solarposition.get_solarposition(weather.index, SITE_LAT, SITE_LON)
print(f"{CITY}: latitude {SITE_LAT:.2f}°, longitude {SITE_LON:.2f}°; weather data year {YEAR}")
print(f"Weather records: {len(weather):,}; building footprints: {len(city_buildings):,}; timestamps use the local time zone.")
display(weather[["GHI", "DNI", "DHI", "Temperature", "Wind Speed"]].head(6))

### 1.1 What Does the City Look Like?

The left map colors buildings by height. The right histogram shows how many buildings fall within each height range.

This practical interprets `height` in meters and uses buildings with valid positive heights for the statistics and shadow calculations.

**Observe:** Are tall buildings clustered together or scattered among lower buildings? Which roofs should be checked first for shading from neighboring buildings?

In [ ]:
height = pd.to_numeric(city_local["height"], errors="coerce")
valid_height = np.isfinite(height) & height.gt(0)
height_buildings = city_local.loc[valid_height].copy()
height_buildings["height"] = height.loc[valid_height]
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
height_buildings.plot(column="height", cmap="viridis", ax=axes[0], legend=True,
    legend_kwds={"label": "Building height (m)", "shrink": .7})
axes[0].set_title(f"{CITY}: building heights and spatial distribution")
axes[0].set_axis_off()
axes[1].hist(height[valid_height], bins=25, color=COLORS[0], edgecolor="white")
axes[1].set(title="Are low-rise or tall buildings more common?", xlabel="Building height (m)", ylabel="Number of footprints")
plt.show()
print(f"Plotted: {valid_height.sum()} footprints with positive heights.")

## 2. Weather: Are Hotter Months Always Sunnier?

Air temperature describes how warm the air is; irradiance describes the intensity of solar radiation received per unit area.

Start with temperature over the year. In the heatmap, the horizontal axis shows the date and the vertical axis shows the time of day. Gray indicates missing data. The plot on the right compares monthly temperature distributions; the line inside each box marks the median.

**Observe:** Does the hottest period occur at noon or later in the afternoon? Which months have a wider temperature range? Compare these patterns with the solar radiation plots below.

In [ ]:
calendar = pd.date_range(weather.index[0].normalize(), weather.index[-1].normalize(), freq="D")
temperature_grid = weather.assign(date=weather.index.normalize(),
    minute=weather.index.hour*60+weather.index.minute).pivot(index="minute", columns="date", values="Temperature")
temperature_grid = temperature_grid.reindex(index=np.arange(0, 1440, STEP_MINUTES), columns=calendar)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
cmap = plt.get_cmap("coolwarm").copy(); cmap.set_bad("#cccccc")
im = axes[0].imshow(temperature_grid, origin="lower", aspect="auto", cmap=cmap,
    extent=(-.5, len(calendar)-.5, 0, 24))
ticks = [i for i, date in enumerate(calendar) if date.day==1]
axes[0].set_xticks(ticks[::2], [f"{calendar[i].month}" for i in ticks[::2]])
axes[0].set(title="Annual temperature: seasons and time of day", ylabel="Local time (hours)")
fig.colorbar(im, ax=axes[0], label="Air temperature (°C)")
axes[1].boxplot([weather.loc[weather.index.month==m, "Temperature"].dropna() for m in range(1,13)], showfliers=False)
axes[1].set(title="Monthly temperature distributions", xlabel="Month", ylabel="Air temperature (°C)")
plt.show()

### 2.1 How Does Solar Radiation Change During a Day?

**Irradiance** is the intensity of solar radiation received per unit area at a given time, measured in W/m².

| Variable | What does it measure? |
|---|---|
| GHI | Global horizontal irradiance |
| DNI | Direct irradiance on a surface perpendicular to the Sun's rays |
| DHI | Diffuse irradiance from the sky on a horizontal surface |

Projecting DNI onto the horizontal plane gives the approximate daytime relationship:

$$GHI \approx DNI\cos(\theta_z)+DHI$$

Here, $\theta_z$ is the solar zenith angle: 0° when the Sun is overhead and 90° at the horizon. The right plot compares the projected estimate with the GHI data.

Start with a complete day. Then change `DAY` to a date in another season and compare daylight duration and curve shape.

In [ ]:
# Select a complete day automatically; alternatively set DAY = "2019-06-15" using the data year.
complete_days = weather.GHI.resample("D").count()
DAY = str(complete_days[complete_days == 24 * 60 / STEP_MINUTES].index[0].date())
day = weather.loc[DAY]
hours = day.index.hour + day.index.minute / 60
horizontal_direct = day.DNI * np.cos(np.deg2rad(solar_position.loc[DAY, "apparent_zenith"])).clip(0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for column, color in zip(["GHI", "DNI", "DHI"], COLORS):
    axes[0].plot(hours, day[column], label=column, color=color)
axes[0].set_title(f"{CITY} · {DAY}: three irradiance components")
axes[1].plot(hours, day.GHI, label="GHI in the data", color=COLORS[0])
axes[1].plot(hours, horizontal_direct + day.DHI, "--", label="Horizontal projection of DNI + DHI", color=COLORS[1])
axes[1].set_title("Project direct irradiance onto the horizontal plane")
for ax in axes:
    ax.set(xlabel="Local time (hours)", ylabel="Irradiance (W/m²)", xlim=(0, 24))
    ax.legend(); ax.grid(alpha=.2)
plt.show()

**Read the plots:** If DNI is the same shortly after sunrise and at noon, is its contribution to a horizontal surface also the same? Explain using the solar zenith angle.

### 2.2 From One Day to a Full Year

The next two plots answer different questions: the heatmap shows when irradiance is strongest, while the monthly bar chart shows the average solar energy received each day.

To convert irradiance to irradiation, multiply by time. Here, each record is treated as a 10-minute mean:

**Irradiation (kWh/m²) = Σ irradiance (W/m²) × (10/60) hours ÷ 1000**

Gray in the heatmap indicates missing data. The monthly bar chart averages daily irradiation over days with complete GHI records.

In [ ]:
# Figure 2: date on the horizontal axis, time on the vertical axis, and GHI as color.
calendar = pd.date_range(weather.index[0].normalize(),
    weather.index[-1].normalize(), freq="D")
heat = weather.assign(date=weather.index.normalize(),
    minute=weather.index.hour * 60 + weather.index.minute).pivot(
        index="minute", columns="date", values="GHI")
heat = heat.reindex(index=np.arange(0, 1440, STEP_MINUTES), columns=calendar)
cmap = plt.get_cmap("magma").copy()
cmap.set_bad("#cccccc")
fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
im = ax.imshow(heat.to_numpy(), origin="lower", aspect="auto", cmap=cmap,
    extent=(-.5, len(calendar)-.5, 0, 24), vmin=0)
month_starts = [i for i, date in enumerate(calendar) if date.day == 1]
ax.set_xticks(month_starts, [f"{calendar[i].month}" for i in month_starts])
ax.set(title=f"{CITY}: daily and seasonal solar radiation patterns", ylabel="Local time (hours)")
fig.colorbar(im, ax=ax, label="GHI（W/m²）")
plt.show()

# Figure 3: calculate daily irradiation, then average complete days by month.
daily_ghi = interval_energy(weather.GHI, STEP_MINUTES).resample("D").sum(min_count=1) / 1000
valid_days = weather.GHI.resample("D").count().eq(24 * 60 / STEP_MINUTES)
monthly_ghi = daily_ghi.where(valid_days).groupby(daily_ghi.index.month).mean().reindex(range(1, 13))
fig, ax = plt.subplots(layout="constrained")
ax.bar(monthly_ghi.index, monthly_ghi, color=COLORS[0])
ax.set(xticks=range(1, 13), xlabel="Month", ylabel="Mean daily irradiation (kWh/m²/day)",
    title="Average daily solar energy by month")
ax.grid(axis="y", alpha=.2)
plt.show()

**Read the plots:** Which month has the highest average daily irradiation? Does its heatmap show longer daylight, stronger irradiance, or both?

## 3. Sun and Shadows: Does Sunshine Always Reach the Roof?

Under the same weather conditions, a low roof beside a tall building may be shaded. Here, building footprints and heights form simplified 3D blocks so that you can observe shadows on the ground and roofs.

**Predict first:** When are shadows usually longest: morning, noon, or evening? Will the same building cast equally long shadows on the ground and on a raised roof?

Run the default settings first. Once the scene appears, move the time slider and drag the scene to rotate the view. The calculation uses the CPU.

| Parameter | What changes? |
|---|---|
| `SHADOW_DATE` | Change the date to compare solar positions across seasons |
| `BLOCK_CENTER_ID` | Choose the central building; `None` selects the tallest building in the data |
| `BLOCK_RADIUS_M` | Set the displayed area; default 80 m, which you can reduce on a slower computer |
| `SHADOW_STEP_MINUTES` | Set the calculation interval; default 30 minutes, with 15 or 60 also available |

This section calculates geometric shading of direct sunlight using flat roofs, level ground, and heights in meters.

In [ ]:
SHADOW_DATE = f"{YEAR}-12-21"  # Change to 06-21 in the same year to compare winter and summer
BLOCK_CENTER_ID = None        # You can also enter a building_id from the table below
BLOCK_RADIUS_M = 80
SHADOW_STEP_MINUTES = 30

# Display IDs that can be used to select another neighborhood.
display(height_buildings[["building_id", "height"]].sort_values("height", ascending=False).head(8))

In [ ]:
from pv_course.course_shadows import build_shadow_scene, display_shadow_scene

shadow_scene = build_shadow_scene(city_buildings, date=SHADOW_DATE,
    center_id=BLOCK_CENTER_ID, radius_m=BLOCK_RADIUS_M,
    step_minutes=SHADOW_STEP_MINUTES)
print(f"Central building:{shadow_scene['center_id']}; showing {len(shadow_scene['buildings'])} buildings.")
shadow_mesh = display_shadow_scene(shadow_scene, CITY)

**Record an observation:** Choose a lower roof and note how its shading differs at two times. Keep the neighborhood unchanged, move the date to June, and rerun the two cells above. Does your explanation still hold?

A building must be taller than the receiving roof to cast a shadow onto it. For the same height difference, a lower Sun produces a longer shadow.

## 4. How Does Tilting a Module Change the Light It Receives?

GHI describes a horizontal surface, while PV modules are often tilted. **POA (plane-of-array) irradiance** is the irradiance reaching the module plane. It has three components:

**POA = direct + sky diffuse + ground-reflected irradiance**

The left plot separates the light received by a fixed module. The right plot compares a horizontal surface, a fixed module, and an ideal dual-axis tracker on the same day. Dual-axis tracking continually adjusts the module to face the Sun.

This example uses the Perez model for POA and a ground albedo of 0.2. Subsequent electricity calculations use a DC model for clean, unshaded modules.

In [ ]:
# Convert city weather into irradiance on the modules, then calculate simulated power.
assessment = run_assessment(CITY, root=ROOT, module_area=MODULE_AREA,
    roof_fraction=ROOF_FRACTION, surface_tilt=SURFACE_TILT,
    surface_azimuth=SURFACE_AZIMUTH)
fixed = assessment["poa_fixed"]
tracking = assessment["poa_tracking"]
power_fixed = assessment["power_fixed"]["power"]
power_tracking = assessment["power_tracking"]["power"]
meta = assessment["metadata"]
print(f"Fixed-system tilt {meta['surface_tilt']:.1f}°, azimuth {meta['surface_azimuth']:.0f}°。")
print(f"Calendar-year data coverage {meta['weather_coverage']['calendar_coverage']:.2%}。")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
components = fixed.loc[DAY]
axes[0].stackplot(hours, components.poa_direct, components.poa_sky_diffuse,
    components.poa_ground, labels=["Direct", "Sky diffuse", "Ground reflected"], colors=COLORS, alpha=.85)
axes[0].set_title("Three POA components for a fixed module")
axes[1].plot(hours, day.GHI, "--", label="Horizontal GHI", color="#777777")
axes[1].plot(hours, fixed.loc[DAY, "total_poa"], label="Fixed mounting", color=COLORS[0])
axes[1].plot(hours, tracking.loc[DAY, "total_poa"], label="Ideal dual-axis tracking", color=COLORS[1])
axes[1].set_title("Same weather, different receiving surfaces")
for ax in axes:
    ax.set(xlabel="Local time (hours)", ylabel="Irradiance (W/m²)", xlim=(0, 24))
    ax.legend(); ax.grid(alpha=.2)
plt.show()

**Read the plots:** Is the difference between tracking and fixed mounting greatest in the morning and evening, or around noon? Does the same conclusion hold on another date?

## 5. From Light to Electricity: Power and Energy

PV modules convert part of the incoming solar energy into electricity. Module power depends on POA and module temperature; module temperature also depends on air temperature and wind speed. We use the supplied module parameters to simulate **the DC power of one module**.

- **Power P (W):** the rate of electricity generation at a given moment.
- **Energy E (kWh):** the total electricity generated over a period.

$$E=\sum_t P_t\Delta t/1000$$

For example, 200 W sustained for 30 minutes produces $200\times0.5/1000=0.1$ kWh. The area under the power curve on the left corresponds to energy. The plot on the right accumulates that energy; its final point is the daily total.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for power, label, color in [(power_fixed, "Fixed mounting", COLORS[0]),
                            (power_tracking, "Ideal dual-axis tracking", COLORS[1])]:
    values = power.loc[DAY]
    axes[0].plot(hours, values, label=label, color=color)
    axes[0].fill_between(hours, values, color=color, alpha=.12)
    # Each record represents the next 10 minutes; cumulative values are placed at interval ends.
    energy = interval_energy(values, STEP_MINUTES).cumsum() / 1000
    axes[1].plot(np.r_[0, hours + STEP_MINUTES/60], np.r_[0, energy], label=label, color=color)
axes[0].set(title="Module power over one day", ylabel="Module DC power (W)")
axes[1].set(title="Cumulative energy over one day", ylabel="Cumulative module DC energy (kWh)")
for ax in axes:
    ax.set(xlabel="Local time (hours)", xlim=(0, 24)); ax.legend(); ax.grid(alpha=.2)
plt.show()

**Read the plots:** Why is cumulative energy constant at night? Compare peak power and total daily energy for the two systems. What does each measure tell you?

Next, aggregate the 10-minute energy values by month and compare the two mounting systems. Percentage gain uses fixed-system energy as the reference:

**Tracking gain = (tracking energy − fixed energy) ÷ fixed energy × 100%**

Monthly energy is summed over the available records. The Shenzhen February data cover 28 days, and the calculation uses those 28 days.

In [ ]:
monthly_energy = pd.DataFrame({
    "Fixed mounting": interval_energy(power_fixed, STEP_MINUTES).resample("MS").sum(min_count=1)/1000,
    "Ideal dual-axis tracking": interval_energy(power_tracking, STEP_MINUTES).resample("MS").sum(min_count=1)/1000})
monthly_energy.index = monthly_energy.index.month
monthly_gain = (monthly_energy["Ideal dual-axis tracking"] / monthly_energy["Fixed mounting"].replace(0, np.nan) - 1) * 100
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
monthly_energy.plot.bar(ax=axes[0], color=COLORS[:2], rot=0)
axes[0].set(title="Monthly module energy", xlabel="Month", ylabel="Module DC energy (kWh)")
axes[1].plot(monthly_gain.index, monthly_gain, "o-", color=COLORS[2])
axes[1].axhline(0, color="#999999", linewidth=.8)
axes[1].set(title="Does tracking gain vary by month?", xticks=range(1, 13), xlabel="Month", ylabel="Energy gain over fixed mounting (%)")
for ax in axes: ax.grid(axis="y", alpha=.2)
plt.show()
summary = assessment["summary"]
fixed_kwh = summary.loc["fixed", "module_kwh"]
tracking_kwh = summary.loc["tracking", "module_kwh"]
print(f"Over the available data period, fixed-module energy is {fixed_kwh:.1f} kWh; tracking-module energy is {tracking_kwh:.1f} kWh。")
print(f"Energy gain over the data period:{(tracking_kwh/fixed_kwh-1):.1%}")

**Read the plots:** Identify the months with the highest and lowest tracking gains. Would the extra electricity justify a tracking mount? What cost or site information would you need?

## 6. From One Module to an Entire Roof

One module has an area of 1.69 m². Divide its energy by its area to obtain energy per square meter of module, then multiply by the module area available on the roof:

**Building energy = module energy ÷ module area × roof area × usable fraction**

The map below shows the fixed-mount scenario, and the histogram shows the distribution of building energy. Roof areas are measured in a local projected coordinate system and multiplied by the usable fraction to obtain module area.

All buildings share the same weather, tilt, and usable fraction, so building energy is proportional to roof area in this model.

In [ ]:
buildings = assessment["buildings"].copy()
map_buildings = buildings.to_crs(buildings.estimate_utm_crs())
map_buildings["energy_mwh"] = map_buildings.annual_energy / 1000
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
map_buildings.plot(column="energy_mwh", ax=axes[0], cmap="viridis", legend=True,
    legend_kwds={"label": "DC energy over the data period (MWh)", "shrink": .65})
axes[0].set_title(f"{CITY}: rooftop energy distribution")
axes[0].set_axis_off()
axes[1].hist(map_buildings.energy_mwh, bins=25, color=COLORS[0], edgecolor="white")
axes[1].set(title="How common are high-energy buildings?", xlabel="Building DC energy over the data period (MWh)", ylabel="Number of buildings")
plt.show()

# Show one building calculation to check areas and units.
example = buildings.loc[buildings.annual_energy.idxmax()]
print(f"Example building ID:{example['building_id']}")
print(f"Roof {example.roof_area:.1f} m² × {ROOF_FRACTION:.0%} = module area {example.usable_area:.1f} m²")
print(f"{fixed_kwh:.1f} kWh ÷ {MODULE_AREA} m² × {example.usable_area:.1f} m² = {example.annual_energy:,.0f} kWh")
audit = assessment["geometry_audit"]
print(f"Removed {audit['exact_duplicates_removed']} exact duplicate footprints; remaining overlap fraction: {audit['overlap_fraction']:.1%}。")
print("Check which buildings overlapping footprints belong to before aggregating regional energy.")

**Read the plots:** Choose a building and calculate its energy if the usable fraction falls from 80% to 50%. By what proportion does the result change? Which model assumption does this reveal?

## 7. Experiment: Does a Steeper Tilt Receive More Solar Energy?

So far, we have used a default tilt. Now keep the city, weather, and orientation fixed, vary only the tilt, and compare POA irradiation over the available data period.

**Predict before running:** Which of the candidate angles below do you expect to receive the most irradiation? Why?

This is a controlled experiment: hold weather and orientation constant to investigate the effect of tilt on incident irradiation (kWh/m²).

In [ ]:
from pv_course.poa_calculation import calculate_poa

TILT_CANDIDATES = [0, 15, 30, 45, 60, 75, 90]
scan_rows = []
for tilt in TILT_CANDIDATES:
    poa = calculate_poa(weather, tilt, SURFACE_AZIMUTH, meta["latitude"], meta["longitude"])
    irradiation = interval_energy(poa["total_poa"], STEP_MINUTES).sum(min_count=1) / 1000
    scan_rows.append({"Tilt (°)": tilt, "POA irradiation over the data period (kWh/m²)": irradiation})
tilt_results = pd.DataFrame(scan_rows).set_index("Tilt (°)")
fig, ax = plt.subplots(layout="constrained")
ax.plot(tilt_results.index, tilt_results.iloc[:, 0], "o-", color=COLORS[0])
ax.axvline(meta["surface_tilt"], linestyle="--", color=COLORS[1], label="Fixed-system tilt used in this practical")
ax.set(title="Vary tilt while keeping orientation fixed", xlabel="Tilt (°)", ylabel="POA irradiation over the data period (kWh/m²)", xticks=TILT_CANDIDATES)
ax.legend(); ax.grid(alpha=.2)
plt.show()
best_tested = tilt_results.iloc[:, 0].idxmax()
print(f"Among these candidate angles,{best_tested}° receives the most irradiation.")
display(tilt_results.round(1))

**Try another step:** Add two angles near the best-performing candidate and rerun the experiment. Does the best candidate change? Then discuss whether the annual result should also be used for a winter-only design.

After choosing a new tilt, update `SURFACE_TILT` in Section 1 and rerun the following sections to calculate the design's energy output.

## 8. Summarize Your Design

Prepare the evidence for **Part A (30 marks)** of your coursework:

1. **Solar resource:** Use a daily or monthly figure and numerical evidence to explain a pattern in your city.
2. **Mounting comparison:** Compare POA and energy for fixed mounting and tracking. Quantify the gain and discuss relevant costs or site conditions.
3. **Rooftop potential:** Include a building map, regional energy total, energy per square meter of module, and one building estimate. Report the footprint overlap information and its effect on the regional estimate.
4. **Parameters and shadows:** Compare a change in tilt or usable roof fraction. Show two shadow times and explain the geometry.

Choose one building and state your proposed tilt, usable roof fraction and estimated energy. Explain an assumption or site condition that could change the estimate.

Save this Notebook with outputs. Include selected evidence in the same report as your independent clustering or forecasting investigation (**Part B, 50 marks**). Conclusions and communication contribute **20 marks**. See [Assignment.md](Assignment.md) for the tasks and [Assessment_Rubric.docx](Assessment_Rubric.docx) for the grading criteria.


In [ ]:
print("Assessment results saved to:", assessment["output_dir"])
print("See Assignment.md for the assignment.")

## Open the Course Results Website

Run the cell below to display your PV assessment and 3D shadows. In the website, click a building to view its area and monthly energy, or open the shadow tab to observe the moving Sun. After changing parameters and recalculating, run this cell again to update the website.

The final cell downloads `PV_Results.zip` in Colab or displays a download link in local Jupyter. Extract it and open `result/course_website.html` in your browser. Save your edited Notebook separately using **File > Download > Download .ipynb** in Colab.

In [ ]:
from course_website import build_website
from pv_course.notebook_outputs import preview_website

website_path = build_website()
preview_website(website_path)


In [ ]:
from pv_course.notebook_outputs import export_results

results_zip = export_results(ROOT)
